# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, CHANNEL, DURATION_SECS, UPLOAD_DATE, LANGUAGE, VIEW_COUNT, EXTRACTION_TIMESTAMP, AUDIO_PATH, TRANSCRIPT_PATH, TRANSCRIPT_STATUS]   

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 [3]. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

In [1]:
# Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Module imports
%%capture
!pip install youtube_transcript_api
# !pip install --upgrade transformers
# !apt-get update
# !apt-get install -y ffmpeg
!pip install yt_dlp webvtt-py
import pandas as pd
import time

In [3]:
#READ DF

df = pd.read_csv('/content/drive/MyDrive/AnnamAI Tasks/Krishi Darshan/YT_DATASET.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14003 entries, 0 to 14002
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   VIDEO_ID              14003 non-null  object 
 1   VIDEO_TITLE           14003 non-null  object 
 2   VIDEO_LINK            14003 non-null  object 
 3   CHANNEL               14003 non-null  object 
 4   DURATION_SECS         13930 non-null  float64
 5   UPLOAD_DATE           0 non-null      float64
 6   LANGUAGE              14003 non-null  object 
 7   VIEW_COUNT            13930 non-null  float64
 8   EXTRACTION_TIMESTAMP  14003 non-null  object 
 9   AUDIO_PATH            0 non-null      float64
 10  TRANSCRIPT_PATH       0 non-null      float64
 11  TRANSCRIPT_STATUS     0 non-null      float64
dtypes: float64(6), object(6)
memory usage: 1.3+ MB


In [4]:
data_lists = {col: df[col].tolist() for col in df.columns}

# To verify, you can print the first few items of each list
for col_name, col_list in data_lists.items():
    print(f"Column '{col_name}': {col_list[:5]}...")

# Create separate list variables for each column from the data_lists dictionary
for col_name, col_list in data_lists.items():
    # Sanitize column names to be valid Python variable names and add '_list' suffix
    variable_name = col_name
    globals()[variable_name] = col_list

Column 'VIDEO_ID': ['SxgaGXcQ8ZY', 'JLwhLr4ZVd0', 'ET8rwerJTg0', 'BfMmAW-58ng', 'VzSq2eW8ZG8']...
Column 'VIDEO_TITLE': ['Krishi Darshan  एकिकृत कृषि प्रणाली', 'Krishi Darshan   Bhagwani ke vikas ka harayana me prayas', 'KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH', 'Krishi Darshan Chana Fasal', 'Krishi Darshan Zaed Phasal new']...
Column 'VIDEO_LINK': ['https://www.youtube.com/watch?v=SxgaGXcQ8ZY', 'https://www.youtube.com/watch?v=JLwhLr4ZVd0', 'https://www.youtube.com/watch?v=ET8rwerJTg0', 'https://www.youtube.com/watch?v=BfMmAW-58ng', 'https://www.youtube.com/watch?v=VzSq2eW8ZG8']...
Column 'CHANNEL': ['Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National']...
Column 'DURATION_SECS': [1377.0, 1472.0, 1491.0, 1221.0, 1476.0]...
Column 'UPLOAD_DATE': [nan, nan, nan, nan, nan]...
Column 'LANGUAGE': ['hi', 'hi', 'hi', 'hi', 'hi']...
Column 'VIEW_COUNT': [7000.0, 1500.0, 915.0, 1400.0, 387.0]...
Column 'EXTRACTION_TIMESTAMP': 

### Attempt to source Transcripts from YT id available

In [5]:
# import yt_dlp

# ydl_opts = {
#     'writeautomaticsub': True,
#     'subtitleslangs': ['hi'],
#     'skip_download': True,
#     'outtmpl': 'extracted_subs',
# }

# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#     ydl.download([video_url])


In [6]:
import os
import re
import yt_dlp
import webvtt

transcripts_fetched = []

def extract_clean_transcript(video_id, language, output_txt_filename="/content/drive/MyDrive/AnnamAI Tasks/Krishi Darshan/Transcripts/"):
    # Temporary paths for the initial download
    temp_dir = "temp_subs"
    os.makedirs(temp_dir, exist_ok=True)

    # 1. Configure yt-dlp to download only the auto-generated subtitle track
    ydl_opts = {
        'writeautomaticsub': True,       # Fetch auto-generated subtitles
        'subtitleslangs': [language],         # Change language code if needed (e.g., 'es')
        'skip_download': True,            # Do NOT download the video file
        'outtmpl': os.path.join(temp_dir, '%(id)s'), # Save file named after the video ID
        'quiet': True,                    # Clean terminal output
    }

    print("The language is ", language, video_id)

    print("Fetching caption stream via yt-dlp...")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download(["https://www.youtube.com/watch?v=" + video_id])
        except Exception as e:
            print(f"Failed downloading caption track: {e}")
            return

    # Find the downloaded file (yt-dlp appends .en.vtt automatically)
    vtt_files = [f for f in os.listdir(temp_dir) if f.endswith('.vtt')]
    if not vtt_files:
        print("No English auto-generated subtitles found for this video.")
        # Clean up empty directory
        os.rmdir(temp_dir)
        return

    downloaded_vtt_path = os.path.join(temp_dir, vtt_files[0])

    # 2. Parse the VTT file and extract *only* text content
    print("Cleaning raw transcript data...")
    lines = []

    for caption in webvtt.read(downloaded_vtt_path):
        # Remove common inline style markers inserted by YouTube auto-captions
        clean_line = re.sub(r'<[^>]*>', '', caption.text)

        # Split text by newlines and trim whitespaces
        for part in clean_line.split('\n'):
            stripped = part.strip()
            if stripped and (not lines or lines[-1] != stripped):
                lines.append(stripped)

    # Combine into a unified continuous text file without line interruptions
    final_raw_text = " ".join(lines)

    # 3. Save purely raw text
    with open(output_txt_filename + video_id + ".txt", "w", encoding="utf-8") as f:
        f.write(final_raw_text)

    # 4. Final Housekeeping: Wipe temporary storage files
    os.remove(downloaded_vtt_path)
    os.rmdir(temp_dir)

    print(f"Finished! Raw caption-only file saved to: {output_txt_filename + video_id + ".txt"}")


In [10]:
#Just testing now

# extract_clean_transcript("xLzJV-K4CSc","hi")

In [ ]:

import multiprocessing as mp

inputs = list(zip(VIDEO_ID, LANGUAGE))
print(len(inputs))

14003


In [ ]:

if __name__ == '__main__':
    process_num = mp.cpu_count()
    print("TOTAL CORES: ", process_num)

    with mp.Pool(processes= process_num) as pool:
        # Slice the video ID list as much as you want
        pool.starmap(extract_clean_transcript, inputs[0:10])


    print("JOB FINISHED")

TOTAL CORES:  2
The language is The language is  hi  hiET8rwerJTg0 
SxgaGXcQ8ZYFetching caption stream via yt-dlp...

Fetching caption stream via yt-dlp...


No English auto-generated subtitles found for this video.
No English auto-generated subtitles found for this video.The language is 
 hiThe language is   JLwhLr4ZVd0hi
 Fetching caption stream via yt-dlp...BfMmAW-58ng

Fetching caption stream via yt-dlp...


No English auto-generated subtitles found for this video.
The language is  hi VzSq2eW8ZG8
Fetching caption stream via yt-dlp...
No English auto-generated subtitles found for this video.
The language is  hi wlwZIC62hXQ
Fetching caption stream via yt-dlp...


No English auto-generated subtitles found for this video.
The language is  hi tqu_7SftoxM
Fetching caption stream via yt-dlp...
No English auto-generated subtitles found for this video.
The language is  hi k3mJBSHc4m4
Fetching caption stream via yt-dlp...


No English auto-generated subtitles found for this video.
The language is  hi _CrkOwCc4pU
Fetching caption stream via yt-dlp...
No English auto-generated subtitles found for this video.


No English auto-generated subtitles found for this video.
The language is  hi 5WlTrocLyL4
Fetching caption stream via yt-dlp...


No English auto-generated subtitles found for this video.
JOB FINISHED


## Whisper implementation for transcription with initial prompts



In [ ]:
## Whisper implementation for transcription with initial prompts

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

In [ ]:
import glob
import os

# Directly target the Krishi Darshan directory
krishi_darshan_dir = "/content/drive/MyDrive/AnnamAI Tasks/Krishi Darshan"

# Use wildcard '*' to represent the YT_video_id folders
search_pattern = os.path.join(krishi_darshan_dir, "*", "audio.mp3")

# Find all paths matching the pattern
audio_paths = glob.glob(search_pattern)

print(f"Total audio files found: {len(audio_paths)}\n")

In [ ]:
def transcribe_with_ai(video_id, audio_file_path):
    result = pipe(
        audio_file_path,
        return_timestamps = True,
        language="hi" #Language is kept as hindi for now due to our project, change according to need 
    )

    with open(f"/content/drive/MyDrive/AnnamAI Tasks/Krishi Darshan/Transcripts/{video_id}.txt", 'w') as tr:
        tr.write(result['text'])

NameError: name 'result' is not defined

In [ ]:
inputs = list(zip(VIDEO_ID, audio_paths))
print(len(inputs))

In [ ]:
if __name__ == '__main__':
    process_num = mp.cpu_count()
    print("TOTAL CORES: ", process_num)

    with mp.Pool(processes= process_num) as pool:
        # Slice the video ID list as much as you want
        pool.starmap(transcribe_with_ai, inputs[:10])


    print("JOB FINISHED")